In [2]:
import pandas as pd
import numpy as np
#read in the data from the csv file
df = pd.read_csv('SGJobData.csv' ,nrows=1200000)


In [3]:
#copy the main data into draft and keep the orignal data intact in df
draft=df.copy()

#draft.head()
#draft.info()
draft[["metadata_expiryDate","metadata_newPostingDate","metadata_originalPostingDate"]].value_counts()
#convert the expiryDate,originalPostingDate,newPostingDate column to datetime format
draft["metadata_expiryDate"] = pd.to_datetime(draft["metadata_expiryDate"], dayfirst=True)
draft["metadata_originalPostingDate"] = pd.to_datetime(draft["metadata_originalPostingDate"], dayfirst=True)
draft["metadata_newPostingDate"] = pd.to_datetime(draft["metadata_newPostingDate"], dayfirst=True)
draft.info()

print("Minimum original posting date:", draft["metadata_originalPostingDate"].min())
print("Maximum original posting date:", draft["metadata_originalPostingDate"].max())
#Date range of the data is from 03-10-2022 to 29-05-2024

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   categories                          1044597 non-null  object        
 1   employmentTypes                     1044597 non-null  object        
 2   metadata_expiryDate                 1044597 non-null  datetime64[ns]
 3   metadata_isPostedOnBehalf           1048585 non-null  bool          
 4   metadata_jobPostId                  1044597 non-null  object        
 5   metadata_newPostingDate             1044597 non-null  datetime64[ns]
 6   metadata_originalPostingDate        1044597 non-null  datetime64[ns]
 7   metadata_repostCount                1048585 non-null  int64         
 8   metadata_totalNumberJobApplication  1048585 non-null  int64         
 9   metadata_totalNumberOfView          1048585 non-null  int64         

In [4]:
draft = draft.drop_duplicates()
draft.duplicated().sum()
#draft["employmentTypes"].value_counts()
#raft["employmentTypes"].isna().sum()
#Unique values in employment types and no NaN values in employment types


0

In [6]:
draft["positionLevels"].value_counts()
draft["positionLevels"].isna().sum()
draft.loc[draft["positionLevels"].isna()]
draft = draft.drop(draft[draft["positionLevels"].isna()].index)
draft["positionLevels"].isna().sum()
#Dropped positionLevels with NaN values


0

In [7]:
draft["salary_type"].value_counts()
draft["salary_type"].isna().sum()
#No NaN values in salary_type

0

In [8]:
draft.describe()
# For rows that have value of less than 100 for maximum,minimum,average salary for monthly salary, replaced the value with NaN as 
# it is not possible to have a salary of 100 for monthly salary. This is done to clean the data and avoid any bias in the analysis.
draft.loc[
    (draft["salary_maximum"] < 100) |
    (draft["salary_minimum"] < 100) |
    (draft["average_salary"] < 100),
    ["salary_maximum", "salary_minimum", "average_salary"]
] = np.nan

# In Cases where the average monthly salary is more than 375,500 for Non executive positions, replaced the value with NaN as it is not possible to have a salary of 375,500 for Non executive positions. This is done to clean the data and avoid any bias in the analysis.
draft.loc[
    (draft["average_salary"] > 375500.0),
    ["salary_maximum", "salary_minimum", "average_salary"]
] = np.nan

#draft["salary_maximum"].describe()
draft["salary_minimum"].describe()
#draft.nlargest(20, "average_salary")


count    1.037163e+06
mean     3.855218e+03
std      3.085934e+03
min      1.000000e+02
25%      2.500000e+03
50%      3.000000e+03
75%      4.500000e+03
max      3.500000e+05
Name: salary_minimum, dtype: float64

In [9]:
#Split the Categories Column into 2 Columns which contains ID and Category
import re

draft["id"] = draft["categories"].str.findall( r'"id"\s*:\s*(\d+)').apply(", ".join)
draft["category"] = draft["categories"].str.findall(r'"category"\s*:\s*"([^"]+)"').apply(", ".join)



In [15]:
#draft.info()
#draft.to_csv("draft.csv", index=False)
#draft["category"].unique()
it_rows = draft[draft["category"].str.contains("Information Technology", na=False)]
it_rows.info()
it_rows["category"].unique()
it_rows.to_csv("it_rows.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 140866 entries, 1 to 1048577
Data columns (total 24 columns):
 #   Column                              Non-Null Count   Dtype         
---  ------                              --------------   -----         
 0   categories                          140866 non-null  object        
 1   employmentTypes                     140866 non-null  object        
 2   metadata_expiryDate                 140866 non-null  datetime64[ns]
 3   metadata_isPostedOnBehalf           140866 non-null  bool          
 4   metadata_jobPostId                  140866 non-null  object        
 5   metadata_newPostingDate             140866 non-null  datetime64[ns]
 6   metadata_originalPostingDate        140866 non-null  datetime64[ns]
 7   metadata_repostCount                140866 non-null  int64         
 8   metadata_totalNumberJobApplication  140866 non-null  int64         
 9   metadata_totalNumberOfView          140866 non-null  int64         
 10  minimum

In [67]:
#Find the Job Category with the most number of Job Postings
job_counts = (
    df.groupby("category")
      .size()
      .reset_index(name="vacancies")
      .sort_values("vacancies", ascending=False)
)

print(job_counts)

KeyError: 'category'

In [10]:
#To find the median salary for a particular Job Category
Category_Summary = (
    df[df["category"] == "Design"] # Edit to include requested Category
      .groupby("positionLevels")
      .agg(
          Vacancies=("average_salary", "count"),
          Median_salary=("average_salary", "median")
      )
      .reset_index()
      .sort_values("Median_salary", ascending=False)
)

print(Category_Summary)

      positionLevels  Vacancies  Median_salary
8  Senior Management         15         9000.0
4  Middle Management         12         6075.0
3            Manager         58         5750.0
6       Professional         75         5750.0
7   Senior Executive        112         5000.0
0          Executive        306         3500.0
2   Junior Executive        179         3250.0
5      Non-executive         70         3000.0
1  Fresh/entry level         87         2400.0


In [4]:
# median_salary = (
#     df.groupby(["category", "positionLevels"])["average_salary"]
#       .median()
#       .reset_index(name="median_salary")
# )

# print(median_salary)

summary = (
    df.groupby(["category", "positionLevels"])
      .agg(
          vacancies=("average_salary", "count"),
          median_salary=("average_salary", "median")
      )
      .reset_index()
      .sort_values("median_salary", ascending=False)
)

print(summary)

                        category     positionLevels  vacancies  median_salary
242   Medical / Therapy Services  Senior Management          8        15250.0
323              Risk Management  Senior Management         52        15000.0
44           Banking and Finance  Senior Management        168        14550.0
197                    Insurance  Senior Management          9        13500.0
368           Telecommunications  Senior Management          4        12750.0
..                           ...                ...        ...            ...
109         Environment / Health  Fresh/entry level         85         2100.0
100                Entertainment  Fresh/entry level         19         2100.0
113         Environment / Health      Non-executive        114         2075.0
145                 General Work  Fresh/entry level        269         1950.0
334  Sciences / Laboratory / R&D  Fresh/entry level        156         1950.0

[387 rows x 4 columns]


In [3]:
df.info()
df["metadata_originalPostingDate"].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83668 entries, 0 to 83667
Data columns (total 23 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   employmentTypes                     83668 non-null  object 
 1   metadata_expiryDate                 83668 non-null  object 
 2   metadata_isPostedOnBehalf           83668 non-null  bool   
 3   metadata_jobPostId                  83668 non-null  object 
 4   metadata_newPostingDate             83668 non-null  object 
 5   metadata_originalPostingDate        83668 non-null  object 
 6   metadata_repostCount                83668 non-null  int64  
 7   metadata_totalNumberJobApplication  83668 non-null  int64  
 8   metadata_totalNumberOfView          83668 non-null  int64  
 9   minimumYearsExperience              83668 non-null  int64  
 10  numberOfVacancies                   83668 non-null  int64  
 11  positionLevels                      83668

count          83668
unique           206
top       2023-03-30
freq            5983
Name: metadata_originalPostingDate, dtype: object